# TMDb Movie Data Analysis

I built this notebook to explore movie patterns across **9,827 movies** from The Movie Database (TMDb). 
My goal was to practice end-to-end data analysis using Python, Pandas, Matplotlib, Seaborn, and SQLite.

### Questions I Wanted to Answer:
1. Which genres appear most often, and what are the most common genre combinations?
2. How did movie releases trend over the years, especially during the 2020 pandemic?
3. How is the catalog divided between English and international languages?
4. What is the true distribution of audience ratings, and are there unrated movies in the dataset?
5. Does high popularity mean a movie will have higher audience ratings?

## 1. Setup & Loading the Data
First, I import the libraries and load the raw CSV file.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Load the dataset
data_path = '../data/mymoviedb.csv' if os.path.exists('../data/mymoviedb.csv') else 'data/mymoviedb.csv'
if not os.path.exists(data_path):
    data_path = 'mymoviedb.csv'

df_raw = pd.read_csv(data_path, lineterminator='\n')
print(f'Loaded dataset: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns')
df_raw.head(3)

## 2. Inspecting the Raw Data
Before jumping into analysis, I inspect the columns, data types, missing values, and check for duplicates.

In [ ]:
print('--- Dataset Info ---')
print(df_raw.info())

print('\n--- Missing Values ---')
print(df_raw.isnull().sum())

print(f'\nExact duplicate rows: {df_raw.duplicated().sum()}')
print(f'Unique movie titles: {df_raw["Title"].nunique()}')
print(f'Repeated title names: {df_raw.duplicated(subset=["Title"]).sum()}')

### Checking Repeated Titles
There are 314 rows where the title name appears more than once. Let's see if these are accidental duplicates or different movies with the same name.

In [ ]:
# Inspect a few repeated titles
repeated_titles = df_raw[df_raw.duplicated(subset=['Title'], keep=False)].sort_values('Title')
repeated_titles[['Title', 'Release_Date', 'Vote_Average', 'Vote_Count']].head(6)

As we can see, movies like *Beauty and the Beast* (1991 vs 2017) or *Halloween* are remakes released in different years. Each row has unique release dates and vote counts, so they are legitimate separate movies.

## 3. Data Cleaning & Dealing with the "Zero-Vote" Anomaly

While checking the summary statistics, I noticed that `Vote_Average` has a minimum value of 0.0.

In [ ]:
# Checking zero-vote records
zero_votes = df_raw[df_raw['Vote_Average'] == 0.0]
print(f'Movies with Vote_Average == 0.0: {len(zero_votes)}')
print(f'Vote_Count values for these movies: {zero_votes["Vote_Count"].unique()}')
zero_votes[['Title', 'Release_Date', 'Genre', 'Popularity']].head(4)

**Why this matters:**
Exactly **100 movies** have 0 votes. These are unreleased, upcoming, or obscure movies that simply have not been reviewed yet.
- If we leave them as 0.0, the average rating for the whole catalog drops from **6.51** to **6.44**.
- If we drop them, we lose 100 real movies from our release year and genre counts.
- The solution: create a column `Rating_Status` to label movies as `'Rated'` or `'Unrated'`.

In [ ]:
df = df_raw.copy()

# 1. Create a surrogate key (movie_id)
df.insert(0, 'movie_id', range(1, len(df) + 1))

# 2. Convert Release_Date to datetime and extract Release_Year
df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')
df['Release_Year'] = df['Release_Date'].dt.year.astype(int)

# 3. Add Rating_Status
df['Rating_Status'] = np.where(df['Vote_Count'] == 0, 'Unrated', 'Rated')

print('Rating status breakdown:')
print(df['Rating_Status'].value_counts())

# Compare raw mean vs. clean mean
raw_mean = df['Vote_Average'].mean()
clean_mean = df[df['Rating_Status'] == 'Rated']['Vote_Average'].mean()
print(f'\nRaw Average Rating (including zeroes): {raw_mean:.2f}')
print(f'Clean Average Rating (rated movies only): {clean_mean:.2f}')

### Splitting Multi-Genre Strings
The `Genre` column contains multiple genres separated by commas (e.g. `'Action, Adventure, Science Fiction'`). 
To analyze genres accurately, I explode this column into a separate table where each row is one movie-genre pair.

In [ ]:
# Create a normalized genre DataFrame
genre_series = df[['movie_id', 'Genre']].copy()
genre_series['genre'] = genre_series['Genre'].str.split(',')
df_genres = genre_series.explode('genre')
df_genres['genre'] = df_genres['genre'].str.strip()
df_genres = df_genres[['movie_id', 'genre']].drop_duplicates()

print(f'Total genre entries: {len(df_genres):,}')
print(f'Unique genres found: {df_genres["genre"].nunique()}')
print('\nTop 5 most frequent genres:')
print(df_genres['genre'].value_counts().head(5))

## 4. Visualizations & Exploratory Analysis

### Chart 1: Top 10 Most Common Genres
Let's see what percentage of movies in the catalog feature each genre.

In [ ]:
genre_counts = df_genres['genre'].value_counts().head(10).reset_index()
genre_counts.columns = ['Genre', 'Count']
genre_counts['Percentage'] = (genre_counts['Count'] / len(df)) * 100.0

plt.figure(figsize=(10, 5))
bars = plt.barh(genre_counts['Genre'][::-1], genre_counts['Percentage'][::-1], color='#2b5c8f')
plt.title('Top 10 Movie Genres by Percentage of Catalog', fontsize=13, fontweight='bold')
plt.xlabel('Percentage of Movies (%)')
plt.ylabel('Genre')

# Add data labels
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.5, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', va='center', fontsize=10)

plt.xlim(0, 45)
plt.tight_layout()
plt.show()

**Takeaway:** Drama leads with 38.1% of all movies, followed by Comedy (30.8%) and Action (27.3%).

### Chart 2: Movie Releases Over Time (1980 to 2022)
How has release volume changed, and what happened during the COVID-19 pandemic?

In [ ]:
releases_recent = df[df['Release_Year'] >= 1980]['Release_Year'].value_counts().sort_index().reset_index()
releases_recent.columns = ['Year', 'Count']

plt.figure(figsize=(11, 4.5))
plt.plot(releases_recent['Year'], releases_recent['Count'], marker='o', markersize=4, color='#1f77b4', linewidth=2)
plt.axvline(2020, color='red', linestyle='--', label='2020 Pandemic Drop (-9.4%)')
plt.axvline(2021, color='green', linestyle=':', label='2021 Backlog Peak (+57.6%)')
plt.title('Annual Movie Releases (1980 - 2022)', fontsize=13, fontweight='bold')
plt.xlabel('Release Year')
plt.ylabel('Movies Released')
plt.legend()
plt.tight_layout()
plt.show()

**Takeaway:** Output grew from 390 titles in 2015 to 500 in 2019 (+28.2%), dropped to 453 in 2020 during filming shutdowns (-9.4%), and then surged to 714 in 2021 (+57.6%) as delayed movies were released.

### Chart 3: Top International (Non-English) Languages
English accounts for 77.0% of titles. Let's see the most common non-English languages.

In [ ]:
lang_counts = df['Original_Language'].value_counts()
non_en = lang_counts.drop('en').head(6).reset_index()
non_en.columns = ['Language', 'Count']
non_en['Share'] = (non_en['Count'] / len(df)) * 100.0

plt.figure(figsize=(9, 4.5))
sns.barplot(data=non_en, x='Language', y='Count', palette='Blues_r')
plt.title('Top 6 Non-English Languages by Movie Count', fontsize=13, fontweight='bold')
plt.xlabel('Language Code')
plt.ylabel('Number of Movies')
for i, row in non_en.iterrows():
    plt.text(i, row['Count'] + 10, f"{row['Count']} ({row['Share']:.1f}%)", ha='center', fontsize=10)
plt.ylim(0, 750)
plt.tight_layout()
plt.show()

**Takeaway:** Japanese (`ja`) is the second most common language at 6.6% (645 titles), driven largely by anime and Japanese cinema, followed by Spanish (`es`, 3.5%) and French (`fr`, 3.0%).

### Chart 4: Audience Rating Distribution (Rated Titles)
How are viewer ratings distributed across rated movies?

In [ ]:
rated_movies = df[df['Rating_Status'] == 'Rated']

plt.figure(figsize=(10, 4.5))
sns.histplot(rated_movies['Vote_Average'], bins=30, kde=True, color='#2c3e50')
plt.axvline(clean_mean, color='red', linestyle='-', label=f'Mean: {clean_mean:.2f}')
plt.axvline(rated_movies['Vote_Average'].median(), color='orange', linestyle='--', label=f'Median: {rated_movies["Vote_Average"].median():.2f}')
plt.title('Audience Rating Distribution (Rated Movies)', fontsize=13, fontweight='bold')
plt.xlabel('Vote Average (0 to 10)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

**Takeaway:** Ratings form a smooth, slightly left-skewed distribution centered around 6.5, with most movies scoring between 5.5 and 7.5.

### Chart 5: Popularity vs. Audience Ratings
Does a higher popularity score mean that audience ratings will also be higher?

In [ ]:
sample = rated_movies.sample(2000, random_state=42)

plt.figure(figsize=(10, 5))
plt.scatter(sample['Vote_Average'], sample['Popularity'], alpha=0.4, color='#3498db', s=25)
plt.yscale('log')
plt.title('Popularity vs. Audience Rating (Log Scale)', fontsize=13, fontweight='bold')
plt.xlabel('Vote Average (0 to 10)')
plt.ylabel('Popularity Score (Log Scale)')
plt.tight_layout()
plt.show()

spearman_corr = rated_movies[['Popularity', 'Vote_Average']].corr(method='spearman').iloc[0, 1]
print(f'Spearman Correlation between Popularity and Rating: {spearman_corr:.3f}')

**Takeaway:** The correlation is weak (+0.127). Highly popular movies are often big franchise blockbusters driven by marketing, which do not always have higher ratings than smaller, well-reviewed movies.

## 5. Key Findings Summary

1. **Catalog Composition:** Drama (38.1%) and Comedy (30.8%) are the most common genres. Action + Adventure is the most frequent genre pair (956 movies).
2. **Release Growth & Shock:** Output expanded +28.2% from 2015 to 2019, dropped by -9.4% in 2020 due to pandemic delays, and surged +57.6% in 2021 to 714 movies.
3. **Language Diversity:** English makes up 77.0% of the catalog. Japanese is second at 6.6%, followed by Spanish at 3.5%.
4. **Rating Quality & Zero-Vote Discovery:** Exactly 100 movies had 0 votes and 0.0 ratings. Labeling them as unrated preserved the catalog size while providing an accurate average rating of 6.51 / 10.
5. **Popularity vs. Rating:** Popularity weakly correlates with ratings ($r \approx 0.13$), showing that marketing buzz and viewer appreciation measure different things.

## 6. What I Learned & Limitations

- **Always check edge cases before cleaning:** Finding the 100 unrated movies showed me why looking at minimum and maximum values is essential before applying transformations.
- **Handling multi-label data:** Normalizing comma-separated genres into a separate table makes analysis much easier and prevents duplicate counting.
- **Limitations:** This dataset reflects TMDb movie metadata, not viewer streaming watch time or subscription numbers. Trends reflect original release dates rather than platform addition dates.